<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/v2s_06_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [2]:
import os
import shutil
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    ReduceLROnPlateau,
    EarlyStopping
)

from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import EfficientNetV2S

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())


TensorFlow version: 2.20.0
GPU: /device:GPU:0


In [3]:
BASE = "/content/newdata"
IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"
MODEL_SAVE_PATH = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/final_model.keras"


In [4]:
mixed_precision.set_global_policy('mixed_float16')

In [5]:
batch_size = 16
image_size = 224

In [6]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

In [7]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)
    image = image / 255.0  # IMPORTANT FIX

    gray = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)

    edge = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge = edge / (tf.reduce_max(edge) + 1e-6)

    return (image, edge), label

In [8]:
def prepare_dataset(path, shuffle):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode='categorical',
        shuffle=shuffle
    )

    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare_dataset(f"{BASE}/train", True)
val_ds = prepare_dataset(f"{BASE}/valid", False)
test_ds = prepare_dataset(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [9]:
def se_block(x, ratio=8):
    filters = x.shape[-1]

    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)

    return layers.Multiply()([x, se])

In [19]:
def create_dual_model():

    # =====================
    # RGB INPUT
    # =====================
    rgb_input = layers.Input(shape=(image_size, image_size, 3), name="rgb_input")

    x_rgb = data_augmentation(rgb_input)
    x_rgb = layers.Rescaling(1./255)(x_rgb)

    # =====================
    # BACKBONE
    # =====================
    base = EfficientNetV2S(include_top=False, weights='imagenet')

    rgb_features = base(x_rgb)

    # =====================
    # EDGE INPUT
    # =====================
    edge_input = layers.Input(shape=(image_size, image_size, 1), name="edge_input")

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(edge_input)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Resizing(rgb_features.shape[1], rgb_features.shape[2])(x)

    # =====================
    # FUSION
    # =====================
    fused = layers.Concatenate()([rgb_features, x])

    fused = layers.Conv2D(256, 3, padding='same', activation='relu')(fused)
    fused = layers.BatchNormalization()(fused)

    fused = se_block(fused)

    fused = layers.Conv2D(512, 3, padding='same', activation='relu')(fused)
    fused = layers.BatchNormalization()(fused)

    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Dropout(0.5)(fused)

    outputs = layers.Dense(2, activation='softmax')(fused)

    # =====================
    # MODEL CREATION (IMPORTANT)
    # =====================
    model = Model(inputs=[rgb_input, edge_input], outputs=outputs)

    # attach backbone AFTER model exists
    model.backbone = base

    # =====================
    # COMPILE
    # =====================
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
        metrics=['accuracy']
    )

    return model

In [20]:
checkpoint_best = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [21]:
model = create_dual_model()
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ edge_input          │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 224, 224,  │        320 │ edge_input[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        128 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ rgb_input[0][0]   │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_3[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_4         │ (None, 224, 224,  │          0 │ sequential[2][0]  │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-s    │ (None, 7, 7,      │ 20,331,360 │ rescaling_4[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing_1          │ (None, 7, 7, 128) │          0 │ batch_normalizat… │
│ (Resizing)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 7, 7,      │          0 │ efficientnetv2-s… │
│ (Concatenate)       │ 1408)             │            │ resizing_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 7, 7, 256) │  3,244,288 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 7, 256) │      1,024 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                 

 Total params: 24,870,146 (94.87 MB)

 Trainable params: 24,714,290 (94.28 MB)

 Non-trainable params: 155,856 (608.81 KB)

In [22]:
print("\n================= STAGE 1 =================")

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[checkpoint_best, reduce_lr]
)


================= STAGE 1 =================
Epoch 1/8
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 535ms/step - accuracy: 0.6998 - loss: 0.5997
Epoch 1: val_accuracy improved from None to 0.65235, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 441s 588ms/step - accuracy: 0.8019 - loss: 0.4917 - val_accuracy: 0.6523 - val_loss: 0.6332 - learning_rate: 1.0000e-04
Epoch 2/8
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - accuracy: 0.8830 - loss: 0.3458
Epoch 2: val_accuracy improved from 0.65235 to 0.88911, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 270s 538ms/step - accuracy: 0.8831 - loss: 0.3385 - val_accuracy: 0.8891 - val_loss: 0.3619 - learning_rate: 1.0000e-04
Epoch 3/8
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 504ms/step - accuracy: 0.8872 - loss: 0.3238
Epoch 3: 

In [23]:
print("\n================= STAGE 2 =================")

# Correct way to get EfficientNet backbone
base_model = None

for layer in model.layers:
    if isinstance(layer, tf.keras.Model) and "efficientnetv2" in layer.name.lower():
        base_model = layer
        break

# safety check
if base_model is None:
    raise ValueError("EfficientNetV2S not found in model")

# unfreeze backbone
base_model.trainable = True

# freeze early layers, fine-tune last layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_best, reduce_lr, early_stop]
)


================= STAGE 2 =================
Epoch 1/10
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - accuracy: 0.9403 - loss: 0.2038
Epoch 1: val_accuracy did not improve from 0.88911
501/501 ━━━━━━━━━━━━━━━━━━━━ 175s 277ms/step - accuracy: 0.9477 - loss: 0.1873 - val_accuracy: 0.8082 - val_loss: 0.5070 - learning_rate: 1.0000e-05
Epoch 2/10
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - accuracy: 0.9381 - loss: 0.1941
Epoch 2: val_accuracy did not improve from 0.88911
501/501 ━━━━━━━━━━━━━━━━━━━━ 138s 276ms/step - accuracy: 0.9486 - loss: 0.1794 - val_accuracy: 0.7822 - val_loss: 0.5369 - learning_rate: 1.0000e-05
Epoch 3/10
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step - accuracy: 0.9453 - loss: 0.1835
Epoch 3: val_accuracy did not improve from 0.88911

Epoch 3: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
501/501 ━━━━━━━━━━━━━━━━━━━━ 132s 262ms/step - accuracy: 0.9521 - loss: 0.1710 - val_accuracy: 0.8741 - val_loss: 0.5000 - learning_rate: 1.0000e-05
Epoch 4/10
501

In [24]:
print("\n================= TEST =================")

loss, acc = model.evaluate(test_ds)
print("Final Test Accuracy:", acc)


================= TEST =================
63/63 ━━━━━━━━━━━━━━━━━━━━ 24s 379ms/step - accuracy: 0.8593 - loss: 0.5085
Final Test Accuracy: 0.8592814207077026


In [25]:
model.save(MODEL_SAVE_PATH)
print("Model Saved!")

Model Saved!
